# Webscraping - Download dos PDFs dos Pareceres do CTC

O objetivo desse código é baixar do site da CVM (https://conteudo.cvm.gov.br/termos_compromisso/index.html) os PDFs referentes aos Pareceres do Comitê de Termo de Compromisso para os processos que resultaram em celebração de TC.

Para isso, usamos um código em Python que baixa os arquivos e salva todos em uma pasta, para que o texto seja extraído posteriormente em outros códigos. Este código foi parcialmente gerado ou assistido por ferramentas de inteligência artificial.

## Instalando os pacotes

In [22]:
pip install selenium

Note: you may need to restart the kernel to use updated packages.


In [1]:
pip install requests beautifulsoup4 tqdm

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import requests
from bs4 import BeautifulSoup
import os
import re
from tqdm import tqdm
import time

## Código

In [30]:
BASE_URL = "https://conteudo.cvm.gov.br"
SEARCH_URL = BASE_URL + "/system/modules/br.com.squadra.principal/elements/resultadoTermoCompromisso.jsp"
PASTA_PDF = r"D:\Backup Tomas\T0mas\Faculdade\IC\Geral\Github CVM\corporate-fraud-in-brazil\data\raw\pdfs\pareceres_do_ctc"

os.makedirs("pareceres", exist_ok=True)

link_texts = [
    'Decisão do Colegiado e Parecer do CTC',
    'Decisão/Parecer',
    'Decisão?Parecer',
    'Decisão /Parecer',
    'Decisáo/Parecer',
    'Decisão do Colegiado',
    'Decisão',
    'Decisão/Parcer'
]

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
    "Accept-Language": "pt-BR,pt;q=0.9,en;q=0.8",
    "Accept-Encoding": "gzip, deflate, br",
    "Connection": "keep-alive",
    "Upgrade-Insecure-Requests": "1",
    "Referer": SEARCH_URL
})

sucessos = 0
falhas = 0

def processar_pagina(soup, num_pagina, processos_falharam):
    """Processar página e baixar PDFs"""
    global sucessos, falhas
    
    table = soup.find('table')
    if not table:
        print(f"❌ Página {num_pagina}: Tabela não encontrada")
        return 0, []
    
    rows = table.find_all('tr')[1:]
    if not rows:
        print(f"❌ Página {num_pagina}: Nenhuma linha de dados")
        return 0, []
    
    print(f"🔍 Página {num_pagina}: {len(rows)} processos encontrados")
    pdfs_baixados = 0
    processos_na_pagina = []
    
    for row in tqdm(rows, desc=f"P{num_pagina:03}"):
        cols = row.find_all('td')
        if len(cols) < 7:
            continue
        
        nome_processo = cols[0].text.strip()
        processos_na_pagina.append(nome_processo)
        nome_arquivo = re.sub(r'[\\/*?:"<>|]', "_", nome_processo)
        
        col_link = cols[-2]
        link_tag = col_link.find('a')
        
        if not link_tag:
            falhas += 1
            processos_falharam.append({
                'nome': nome_processo,
                'pagina': num_pagina,
                'motivo': 'Link não encontrado na tabela',
                'url_detalhes': 'N/A'
            })
            continue
        
        texto_link = link_tag.text.strip().lower()
        if not any(x.lower() in texto_link for x in link_texts):
            falhas += 1
            processos_falharam.append({
                'nome': nome_processo,
                'pagina': num_pagina,
                'motivo': f'Tipo de link não corresponde: "{texto_link}"',
                'url_detalhes': 'N/A'
            })
            continue
        
        detalhe_url = link_tag['href']
        if not detalhe_url.startswith("http"):
            detalhe_url = BASE_URL + detalhe_url
        
        resultado_download = baixar_pdf_detalhado(nome_arquivo, detalhe_url)
        
        if resultado_download['sucesso']:
            pdfs_baixados += 1
            sucessos += 1
        else:
            falhas += 1
            processos_falharam.append({
                'nome': nome_processo,
                'pagina': num_pagina,
                'motivo': resultado_download['motivo'],
                'url_detalhes': detalhe_url
            })
    
    return pdfs_baixados, processos_na_pagina

def baixar_pdf_detalhado(nome_arquivo, detalhe_url):
    """
    Baixar PDF de um processo com detalhamento do motivo de falha
    Retorna dict com 'sucesso' e 'motivo'
    """
    try:
        detalhe_resp = session.get(detalhe_url, timeout=10)
        
        if detalhe_resp.status_code != 200:
            return {
                'sucesso': False,
                'motivo': f'Erro HTTP {detalhe_resp.status_code} ao acessar página de detalhes'
            }
        
        detalhe_soup = BeautifulSoup(detalhe_resp.text, 'html.parser')
        
        pdf_tag = detalhe_soup.find('a', href=lambda href: href and href.lower().endswith('.pdf'))
        if not pdf_tag:
            return {
                'sucesso': False,
                'motivo': 'Link do PDF não encontrado na página de detalhes'
            }
        
        pdf_url = pdf_tag['href']
        if pdf_url.startswith("/"):
            pdf_url = BASE_URL + pdf_url
        
        pdf_response = session.get(pdf_url, timeout=30)
        
        if pdf_response.status_code != 200:
            return {
                'sucesso': False,
                'motivo': f'Erro HTTP {pdf_response.status_code} ao baixar PDF'
            }
        
        pdf_content = pdf_response.content
        
        if not pdf_content:
            return {
                'sucesso': False,
                'motivo': 'PDF vazio ou não recebido'
            }
        
        if not pdf_content.startswith(b'%PDF'):
            return {
                'sucesso': False,
                'motivo': 'Arquivo baixado não é um PDF válido'
            }
        
        # Salvar arquivo
        arquivo_path = os.path.join(PASTA_PDF, f"{nome_arquivo}.pdf")
        contador = 1
        while os.path.exists(arquivo_path):
            arquivo_path = os.path.join(PASTA_PDF, f"{nome_arquivo}_{contador}.pdf")
            contador += 1
        
        with open(arquivo_path, 'wb') as f:
            f.write(pdf_content)
        
        return {
            'sucesso': True,
            'motivo': 'PDF baixado com sucesso'
        }
        
    except requests.exceptions.Timeout:
        return {
            'sucesso': False,
            'motivo': 'Timeout ao baixar PDF'
        }
    except requests.exceptions.RequestException as e:
        return {
            'sucesso': False,
            'motivo': f'Erro de conexão: {str(e)}'
        }
    except Exception as e:
        return {
            'sucesso': False,
            'motivo': f'Erro inesperado: {str(e)}'
        }

def navegar_para_pagina(num_pagina):
    """
    Navegar para uma página específica usando múltiplas estratégias baseadas no HTML
    """
    print(f"🔄 Navegando para página {num_pagina}...")
    
    # ESTRATÉGIA 1: Usar formulário oculto formParam
    estrategias = [
        {
            "nome": "FormParam (formulário oculto)",
            "data": {
                "searchPage": str(num_pagina),
                "lastName": "",
                "itensPagina": "5",
                "max": "169"
            }
        },
        
        # ESTRATÉGIA 2: Simular "Ir para a página"
        {
            "nome": "Ir para a página",
            "data": {
                "irPara": str(num_pagina),
                "max": "169"
            }
        },
        
        # ESTRATÉGIA 3: Usar parâmetros de paginação padrão
        {
            "nome": "Paginação padrão",
            "data": {
                "pagina": str(num_pagina),
                "max": "169"
            }
        },
        
        # ESTRATÉGIA 4: Simular botão com valor específico
        {
            "nome": "Botão simulado",
            "data": {
                str(num_pagina): "",
                "max": "169"
            }
        },
        
        # ESTRATÉGIA 5: Combinação de parâmetros
        {
            "nome": "Combinação completa",
            "data": {
                "searchPage": str(num_pagina),
                "pagina": str(num_pagina),
                "irPara": str(num_pagina),
                "max": "169",
                "itensPagina": "5"
            }
        }
    ]
    
    for i, estrategia in enumerate(estrategias):
        try:
            print(f"  🎯 Tentativa {i+1}: {estrategia['nome']}")
            
            # Tentar POST primeiro
            response = session.post(SEARCH_URL, data=estrategia['data'], timeout=15)
            
            if response.status_code == 200:
                response.encoding = 'utf-8'
                soup = BeautifulSoup(response.text, 'html.parser')
                
                # Verificar se a navegação funcionou
                if verificar_pagina_correta(soup, num_pagina):
                    print(f"  ✅ Sucesso com {estrategia['nome']}!")
                    return soup
            
            # Se POST não funcionar, tentar GET
            response = session.get(SEARCH_URL, params=estrategia['data'], timeout=15)
            
            if response.status_code == 200:
                response.encoding = 'utf-8'
                soup = BeautifulSoup(response.text, 'html.parser')
                
                if verificar_pagina_correta(soup, num_pagina):
                    print(f"  ✅ Sucesso com {estrategia['nome']} (GET)!")
                    return soup
            
        except Exception as e:
            print(f"  ❌ {estrategia['nome']} falhou: {e}")
            continue
    
    print(f"  ❌ Todas as estratégias falharam para página {num_pagina}")
    return None

def verificar_pagina_correta(soup, num_pagina_esperada):
    """
    Verificar se estamos na página correta
    """
    try:
        # Procurar pelo indicador "Páginas X/169"
        span_pagina = soup.find('span', string=re.compile(r'Páginas? \d+/\d+'))
        if span_pagina:
            match = re.search(r'Páginas? (\d+)/\d+', span_pagina.text)
            if match:
                pagina_atual = int(match.group(1))
                return pagina_atual == num_pagina_esperada
        
        # Se não encontrar o indicador, assumir que funcionou
        # (algumas páginas podem não ter esse indicador)
        table = soup.find('table')
        return table is not None
        
    except Exception as e:
        print(f"  ⚠️ Erro ao verificar página: {e}")
        return False

def main():
    """Função principal usando navegação direta para páginas específicas"""
    print("🚀 Iniciando scraper CVM (navegação direta por página)...")
    
    # Lista para armazenar processos que falharam
    processos_falharam = []
    
    # Fazer requisição inicial para configurar sessão
    try:
        print("🌐 Configurando sessão inicial...")
        response = session.get(SEARCH_URL)
        response.encoding = 'utf-8'
        print("✅ Sessão configurada")
    except Exception as e:
        print(f"❌ Erro ao configurar sessão: {e}")
        return
    
    total_paginas = 169
    paginas_processadas = 0
    
    # SEMPRE processar TODAS as 169 páginas
    for num_pagina in range(1, total_paginas + 1):
        print(f"\n📄 === PÁGINA {num_pagina}/{total_paginas} ===")
        
        # Navegar para a página específica
        soup = navegar_para_pagina(num_pagina)
        
        if not soup:
            print(f"❌ Falha ao acessar página {num_pagina} - continuando...")
            continue
        
        # Processar página atual
        pdfs_baixados, processos = processar_pagina(soup, num_pagina, processos_falharam)
        
        if pdfs_baixados > 0:
            paginas_processadas += 1
            print(f"📊 PDFs baixados: {pdfs_baixados}")
            print(f"🔍 Primeiro processo: {processos[0][:60]}..." if processos else "")
        else:
            print(f"⚠️ Nenhum PDF baixado na página {num_pagina} - continuando...")
        
        # Status geral
        print(f"📈 Total baixados: {sucessos} | Falhas: {falhas}")
        print(f"📄 Páginas processadas: {paginas_processadas}")
        
        # Pausa entre páginas
        time.sleep(1.5)
    
    # Salvar lista de processos que falharam
    if processos_falharam:
        print(f"\n💾 Salvando lista de {len(processos_falharam)} processos que falharam...")
        
        with open("processos_falharam.txt", "w", encoding="utf-8") as f:
            f.write("PROCESSOS QUE FALHARAM - PDF NÃO EXTRAÍDO\n")
            f.write("=" * 50 + "\n\n")
            
            for i, processo in enumerate(processos_falharam, 1):
                f.write(f"{i:03d}. {processo['nome']}\n")
                f.write(f"     Página: {processo['pagina']}\n")
                f.write(f"     Motivo: {processo['motivo']}\n")
                f.write(f"     URL Detalhes: {processo['url_detalhes']}\n\n")
        
        print(f"✅ Lista salva em: processos_falharam.txt")
    
    print(f"\n🎯 SCRAPING CONCLUÍDO!")
    print(f"✅ Total de PDFs baixados: {sucessos}")
    print(f"❌ Total de falhas: {falhas}")
    print(f"📄 Páginas processadas: {paginas_processadas}/{total_paginas}")
    print(f"💾 PDFs salvos em: ./pareceres/")
    print(f"📋 Lista de falhas salva em: processos_falharam.txt")

if __name__ == "__main__":
    main()

🚀 Iniciando scraper CVM (navegação direta por página)...
🌐 Configurando sessão inicial...
✅ Sessão configurada

📄 === PÁGINA 1/169 ===
🔄 Navegando para página 1...
  🎯 Tentativa 1: FormParam (formulário oculto)
  ✅ Sucesso com FormParam (formulário oculto)!
🔍 Página 1: 5 processos encontrados


P001: 100%|██████████████████████████████████████████████████████████████████████████████| 5/5 [00:04<00:00,  1.22it/s]


📊 PDFs baixados: 5
🔍 Primeiro processo: 19957.004339/2023-17...
📈 Total baixados: 5 | Falhas: 0
📄 Páginas processadas: 1

📄 === PÁGINA 2/169 ===
🔄 Navegando para página 2...
  🎯 Tentativa 1: FormParam (formulário oculto)
  ✅ Sucesso com FormParam (formulário oculto)!
🔍 Página 2: 5 processos encontrados


P002: 100%|██████████████████████████████████████████████████████████████████████████████| 5/5 [00:03<00:00,  1.38it/s]


📊 PDFs baixados: 5
🔍 Primeiro processo: 19957.008609/2024-31...
📈 Total baixados: 10 | Falhas: 0
📄 Páginas processadas: 2

📄 === PÁGINA 3/169 ===
🔄 Navegando para página 3...
  🎯 Tentativa 1: FormParam (formulário oculto)
  ✅ Sucesso com FormParam (formulário oculto)!
🔍 Página 3: 5 processos encontrados


...

⚠️ Nenhum PDF baixado na página 168 - continuando...
📈 Total baixados: 745 | Falhas: 95
📄 Páginas processadas: 160



📄 === PÁGINA 169/169 ===
🔄 Navegando para página 169...
  🎯 Tentativa 1: FormParam (formulário oculto)
  ✅ Sucesso com FormParam (formulário oculto)!
🔍 Página 169: 3 processos encontrados


P169: 100%|████████████████████████████████████████████████████████████████████████████| 3/3 [00:00<00:00, 8371.86it/s]

⚠️ Nenhum PDF baixado na página 169 - continuando...
📈 Total baixados: 745 | Falhas: 98
📄 Páginas processadas: 160



💾 Salvando lista de 98 processos que falharam...
✅ Lista salva em: processos_falharam.txt

🎯 SCRAPING CONCLUÍDO!
✅ Total de PDFs baixados: 745
❌ Total de falhas: 98
📄 Páginas processadas: 160/169
💾 PDFs salvos em: ./pareceres/
📋 Lista de falhas salva em: processos_falharam.txt
